# 01 — Introduction to DrakonixBacktester

**Goal:** Run your first backtests using the DrakonixBacktester framework and compare three strategies on SPY.

---

## The Framework

DrakonixBacktester has four components:

| Module | Purpose |
|---|---|
| `engine.py` | `Backtester` runs the simulation; `BacktestResult` holds output |
| `strategy.py` | Abstract `Strategy` base class — implement `generate_signal()` |
| `metrics.py` | Sharpe ratio, max drawdown, CAGR, etc. |
| `strategies/` | Built-in strategies: `BuyAndHold`, `SMACrossover`, `BollingerBands` |

**Execution model:** Signal at close of day T → executes at close of day T+1.  
This avoids look-ahead bias — you can only act on information you actually had.

In [ ]:
import sys
sys.path.insert(0, '../../')  # make DrakonixBacktester importable from lessons/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import warnings

from DrakonixBacktester import Backtester
from DrakonixBacktester.strategies import BuyAndHold, SMACrossover, BollingerBands
from DrakonixBacktester import metrics

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (13, 7)

In [ ]:
# Download SPY data
spy = yf.download('SPY', start='2015-01-01', end='2025-01-01', auto_adjust=True)
prices = spy['Close'].squeeze().dropna()

print(f'{len(prices)} trading days ({prices.index[0].date()} to {prices.index[-1].date()})')

---

## 1. Baseline: Buy and Hold

Always run buy-and-hold first. It's your benchmark — any active strategy that can't beat it isn't worth running (after costs).

In [ ]:
bh = Backtester(prices=prices, strategy=BuyAndHold(), initial_capital=10_000)
bh_result = bh.run()

print('Buy and Hold')
print(bh_result.summary())
bh_result.plot(title='Buy and Hold — SPY 2015–2025')

---

## 2. Momentum: SMA Crossover (20/50)

Buy when the 20-day SMA crosses above the 50-day SMA (uptrend starting).  
Sell when it crosses below (downtrend starting). Otherwise hold.

In [ ]:
sma = Backtester(
    prices=prices,
    strategy=SMACrossover(fast=20, slow=50),
    initial_capital=10_000,
)
sma_result = sma.run()

print(f'Number of trades: {len(sma_result.trades)}')
print()
print('SMA Crossover (20/50)')
print(sma_result.summary())
sma_result.plot(title='SMA Crossover (20/50) — SPY 2015–2025', benchmark=bh_result.equity)

---

## 3. Mean Reversion: Bollinger Bands (20-day, 2σ)

Buy when price touches the lower band (statistically oversold).  
Sell when price returns to the 20-day moving average.

In [ ]:
bb = Backtester(
    prices=prices,
    strategy=BollingerBands(window=20, num_std=2.0),
    initial_capital=10_000,
)
bb_result = bb.run()

print(f'Number of trades: {len(bb_result.trades)}')
print()
print('Bollinger Bands (20, 2σ)')
print(bb_result.summary())
bb_result.plot(title='Bollinger Bands (20, 2σ) — SPY 2015–2025', benchmark=bh_result.equity)

---

## 4. Head-to-Head Comparison

In [ ]:
comparison = pd.DataFrame({
    'Buy & Hold':     bh_result.summary(),
    'SMA Cross 20/50': sma_result.summary(),
    'Bollinger 20/2σ': bb_result.summary(),
})
print(comparison.to_string())

In [ ]:
# Equity curves on one chart (all normalized to $10,000 start)
fig, ax = plt.subplots(figsize=(13, 6))

bh_result.equity.plot(ax=ax, label='Buy & Hold', color='gray', linestyle='--', linewidth=1.5)
sma_result.equity.plot(ax=ax, label='SMA Crossover 20/50', color='steelblue', linewidth=1.5)
bb_result.equity.plot(ax=ax, label='Bollinger Bands 20/2σ', color='darkorange', linewidth=1.5)

ax.set_title('Strategy Comparison — SPY 2015–2025')
ax.set_ylabel('Portfolio Value ($)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend()
plt.tight_layout()
plt.show()

---

## 5. Writing Your Own Strategy

Subclass `Strategy` and implement `generate_signal()`. Here's a minimal example — a simple **RSI mean reversion** strategy:

In [ ]:
from DrakonixBacktester import Strategy


class RSIMeanReversion(Strategy):
    """
    Buy when RSI drops below oversold threshold.
    Sell when RSI recovers above the midpoint (50).
    """

    def __init__(self, window: int = 14, oversold: float = 30.0):
        self.window = window
        self.oversold = oversold
        self._in_trade = False

    def _rsi(self, prices: pd.Series) -> float:
        delta = prices.diff().dropna()
        gain = delta.clip(lower=0).rolling(self.window).mean()
        loss = (-delta.clip(upper=0)).rolling(self.window).mean()
        rs = gain / loss.replace(0, float('inf'))
        return float(100 - (100 / (1 + rs.iloc[-1])))

    def generate_signal(self, prices: pd.Series) -> int:
        if len(prices) < self.window + 1:
            return 0

        rsi = self._rsi(prices)

        if not self._in_trade and rsi < self.oversold:
            self._in_trade = True
            return 1

        if self._in_trade and rsi >= 50:
            self._in_trade = False
            return -1

        return 0

    def reset(self):
        self._in_trade = False


rsi_bt = Backtester(prices=prices, strategy=RSIMeanReversion(window=14, oversold=30))
rsi_result = rsi_bt.run()

print(f'Number of trades: {len(rsi_result.trades)}')
print()
print('RSI Mean Reversion (14, oversold=30)')
print(rsi_result.summary())
rsi_result.plot(title='RSI Mean Reversion — SPY 2015–2025', benchmark=bh_result.equity)

---

## Key Takeaways

1. **Always benchmark against buy-and-hold first.** Most strategies underperform it after costs.
2. **Trend-following vs. mean reversion are philosophically opposite** — they tend to have different failure modes. SMA crossover struggles in choppy markets; Bollinger Bands struggles in trending markets.
3. **A good backtest result ≠ a good strategy.** The next notebook covers the pitfalls: overfitting, look-ahead bias, and survivorship bias.
4. **Implementing your own strategy is just one method.** The `generate_signal(prices)` contract is the only thing the backtester requires.

## Next

**`02_backtesting_pitfalls.ipynb`** — How to fool yourself with a backtest, and how not to.